[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/03_cluster.ipynb)

# Shared topic space — embed, freeze, label, lens

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-16

**Abstract**: Once phrases exist, we place them in one embedding space and partition with
$k$-means. Two design choices matter more than the brand of embedder: frozen
assignments when the phrase set is unchanged, and blind labels and lenses
proposed without seeing group labels. This notebook teaches those choices and
shows a PCA view as a diagnostic — not as the generative story for the phrases.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

Hard clustering gives every phrase a single topic label. Multilogue embeds
phrases, runs $k$-means, and optionally freezes the assignment vector so later
notebooks do not silently reshuffle topics when $k$-means initialisation differs
across machines.

Labels name clusters; lenses group clusters into coarser framings (economic,
social, …). Both are proposed from example phrases alone — *format-blind* or
*group-blind* — so the taxonomy is not fitted to the comparison we later want
to make.


## Frozen assignments

$k$-means is sensitive to initialisation and is not bit-stable across machines.
When the phrase set (and fingerprint) is unchanged we reuse saved assignments.
That is a reproducibility device: the scientific claim is “these phrases under
this embedding and $k$”, not “whatever $k$-means happened to return today”.


## PCA as a viewing device

The scatter below projects embeddings onto the first two principal components.
Classical PCA finds linear directions of high variance; it is a *view*, not the
generative model for the phrases. Soft lens memberships under probabilistic PCA
and $\sigma^2$ chosen by earth mover's distance live in notebook 08.


In [ ]:
from IPython.display import display
import numpy as np
from multilogue.access import load
from multilogue.dummy import FakeClient
from multilogue.extract import make_extract_work, phrase_table
from multilogue.llm import StageRunner
from multilogue.cluster import (
    Embedder, frozen_or_new_assignments, example_phrases_for_clusters,
    make_label_work, make_lens_work, resolve_lenses,
)
from multilogue import viz


In [ ]:
units = load("synthetic", n_docs_per_group=3, units_per_doc=3)["units"]
units_by_id = {u["unit_id"]: u for u in units}
client = FakeClient()
root = FIG.parent
runner = StageRunner(root / "ckpt_cl", root / "logs_cl", max_workers=2, sleep=lambda _: None)
extraction = {}
for kind in ("concern", "benefit"):
    work = make_extract_work(
        client, kind=kind, system="e",
        user_for_unit=lambda u, k=kind: u["text"], model="fake-small",
        sentinel=f"NO_{kind.upper()}",
    )
    extraction[kind] = runner.run(
        f"ex_{kind}", units, lambda u: u["unit_id"], work, f"ex|{kind}", quiet=True,
    )
phrases = phrase_table(
    extraction, units_by_id, kinds=["concern", "benefit"], extra_unit_fields=["group"],
)
k = 4
embedder = Embedder(root / "emb", client, model="fake-emb")
E = np.asarray(embedder(phrases.phrase.tolist()), float)
labels, centroids, frozen = frozen_or_new_assignments(
    phrases, E, k, 0, root / "clusters",
)
phrases = phrases.copy()
phrases["cluster"] = labels
phrases["centroid_sim"] = (E * centroids[labels]).sum(1)
print("frozen_assignments:", frozen)


In [ ]:
viz.cluster_size_bars(phrases, path=FIG / "03_cluster_sizes.png")
viz.embedding_scatter(E, phrases["cluster"], path=FIG / "03_pca_clusters.png")


Cluster sizes show whether $k$ produced empty or tiny topics. The PCA scatter
colours phrases by hard cluster. Hopeless overlap suggests revisiting $k$ or the
phrase set before interpreting shares.


## Blind labels and lenses

Labels name clusters; lenses group them. Researchers may edit the lens mapping
afterwards — the point of the first pass is a blind proposal.


In [ ]:
cluster_units = example_phrases_for_clusters(phrases, k, 0)
lab = runner.run(
    "labels", cluster_units, lambda cu: cu["key"],
    make_label_work(
        client, system="label",
        prompt_template="Label these:\n<<PHRASES>>", model="fake-small",
    ),
    "labels|v1", quiet=True,
)
names = {
    cu["cluster"]: lab.get(cu["key"], {}).get("label", f"cluster {cu['cluster']}")
    for cu in cluster_units
}
print(names)
cl_desc = "\n".join(
    f"{c}: {names[c]} — ({int((phrases.cluster == c).sum())})" for c in range(k)
)
lens_runs = runner.run(
    "lenses", [1, 2], lambda r: str(r),
    make_lens_work(
        client, system="lens", prompt_template="Group:\n<<CLUSTERS>>",
        cluster_description=cl_desc, model="fake-small", k=k,
    ),
    f"lenses|{cl_desc[:60]}", quiet=True,
)
mapping, edited = resolve_lenses(k, lens_runs)
phrases["lens"] = phrases["cluster"].map(mapping)
print("lens mapping:", mapping, "edited:", edited)


In [ ]:
viz.embedding_scatter(
    E, phrases["lens"],
    title="Embedding space coloured by lens (PCA view)",
    path=FIG / "03_pca_lenses.png",
)


# Outlook

Hard lens labels are useful but brittle. Notebook 04 turns cluster or lens
assignments into *document-weighted* shares so long documents do not drown short
ones. Notebook 08 softens lenses with a mixture of probabilistic PCAs.
